# Análisis RF Virtual · Mini Penal A · UP N°8 Piñero

Pipeline Sionna RT 2.x sobre la geometría de Piñero, con calibración transferida desde Recreo.

- **Cliente**: Motorola Solutions Argentina — CP 01/26
- **Compute**: Google Colab Pro con GPU T4 (~1-1.5 h)
- **Entrada**: `pipeline_rf/inputs/` del repo (clonado automáticamente en Cell 2)
- **Salida**: `/content/outputs/` (storage efímero Colab; zip descargable en Cell 12)
- **Repo**: https://github.com/MollytheCatLoca/RFQ_MOTOROLA_IMSI

**Corridas programadas**: 3 bandas (B26/B28/B41) × 2 alturas (1.5 m / 7.5 m) = 6 radio maps

## Cell 1 · Install Sionna RT + deps

In [ ]:
!pip install -q sionna==0.19.2 mitsuba drjit
!pip install -q tensorflow==2.15.0 matplotlib pandas
print('✅ Sionna + deps instalados')

## Cell 2 · Clonar repo desde GitHub + setup paths

Colab clona el repo público de BIS (o privado con OAuth si aplica).
Los outputs se generan en `/content/outputs/` (storage efímero Colab) y se bajan con `files.download()` al final.

In [ ]:
import os

# Clonar repo (público) · si es privado Colab va a pedir OAuth GitHub transparente
REPO_URL = 'https://github.com/MollytheCatLoca/RFQ_MOTOROLA_IMSI.git'
REPO_DIR = '/content/RFQ_MOTOROLA_IMSI'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

INPUTS = f'{REPO_DIR}/pipeline_rf/inputs'
OUTPUTS = '/content/outputs'
os.makedirs(f'{OUTPUTS}/heatmaps', exist_ok=True)

# Verificar inputs clonados
expected = ['PINERO_UP8_scene.xml', 'UP9_unidad_alto_perfil.obj', 'UP9_unidad_alto_perfil.mtl',
            'UP9_RF_manifest.json', 'UP9_objects_rf.csv', 'material_mapping.json',
            'antenas_pinero_enriched.csv', 'parametros_calibrados.py']
missing = [f for f in expected if not os.path.exists(f'{INPUTS}/{f}')]
assert not missing, f'Faltan archivos en inputs/: {missing}'
print(f'✅ Repo clonado · {len(expected)} inputs OK · outputs en {OUTPUTS}')

## Cell 3 · Imports + cargar parámetros calibrados (F1)

In [ ]:
import sys, json, math, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Agregar inputs al path para importar parametros_calibrados
sys.path.insert(0, INPUTS)
import parametros_calibrados as cfg

import sionna.rt as rt
print(f'Sionna RT version: {rt.__version__ if hasattr(rt, "__version__") else "unknown"}')

print(f'Bandas prioritarias: {cfg.SYSTEMIC_PRIORITY_BANDS}')
print(f'Alturas analíticas: {cfg.SIONNA_ANALYSIS_HEIGHTS_M} m')
print(f'NLOS offset ITU-R M.2412: +{cfg.SYSTEMIC_NLOS_OFFSET_DB} dB')

## Cell 4 · Cargar escena Mitsuba + asignar radio materials ITU-R P.2040

Los 2724 objetos del manifest se clasifican en 5 materiales RF (concrete, metal, glass, ground_dry, ground_wet). Sionna trae materiales ITU-R incorporados como `itu_concrete`, `itu_metal`, `itu_glass`, `itu_wet_ground`, `itu_dry_ground`.

In [ ]:
# Cargar escena
scene = rt.load_scene(f'{INPUTS}/PINERO_UP8_scene.xml')
scene.frequency = cfg.SYSTEMIC_BAND_FREQ_MHZ[cfg.SYSTEMIC_PRIORITY_BANDS[0]] * 1e6
print(f'Escena cargada · freq inicial: {scene.frequency/1e6:.1f} MHz')

# Cargar mapping de materials
with open(f'{INPUTS}/material_mapping.json') as f:
    mat_map = json.load(f)

# Sionna name mapping
SIONNA_MAT = {
    'concrete':   'itu_concrete',
    'brick':      'itu_brick',
    'glass':      'itu_glass',
    'metal':      'itu_metal',
    'ground_wet': 'itu_wet_ground',
    'ground_dry': 'itu_medium_dry_ground',
    'wood':       'itu_wood',
}

# Asignar material a cada objeto presente en la escena
assigned = 0; skipped = 0
for obj_name, rf_mat in mat_map.items():
    sionna_mat = SIONNA_MAT.get(rf_mat, 'itu_concrete')
    try:
        obj = scene.get(obj_name)
        if obj is not None:
            obj.radio_material = sionna_mat
            assigned += 1
        else:
            skipped += 1
    except Exception:
        skipped += 1
print(f'✅ Materials asignados: {assigned} objetos · skipped: {skipped}')

## Cell 5 · Cargar antenas + aplicar co-ubicación UMTS/LTE

Assumption documentado en F2 README: los eNodeB LTE AR se montan sobre la infraestructura UMTS existente. Para las 3 bandas LTE prioritarias (B26/B28/B41) se usan **todas** las antenas del operador correspondiente (UMTS + LTE + GSM si aplica), no solo las reportadas como `radio == 'LTE'` en OpenCellID.

In [ ]:
df_ant = pd.read_csv(f'{INPUTS}/antenas_pinero_enriched.csv')
print(f'Antenas totales: {len(df_ant)}')
print(df_ant['operator'].value_counts())

# Filtrar antenas dentro de 5 km del centroide UP8 (más allá contribuyen poco)
df_ant_5km = df_ant[df_ant['distance_km_to_up8'] <= 5.0].copy()
print(f'\nAntenas dentro 5 km: {len(df_ant_5km)}')

# Convertir lat/lon → coordenadas locales de la escena (metros desde centroide)
UP8_LAT = -33.09270383406942
UP8_LON = -60.80047176165265
M_PER_DEG_LAT = 111320.0
def m_per_deg_lon(lat):
    return 111320.0 * math.cos(math.radians(lat))

df_ant_5km['x_m'] = (df_ant_5km['lon'] - UP8_LON) * m_per_deg_lon(UP8_LAT)
df_ant_5km['y_m'] = (df_ant_5km['lat'] - UP8_LAT) * M_PER_DEG_LAT
df_ant_5km['z_m'] = cfg.SYSTEMIC_H_TX_MACRO_M  # 25 m típica macro

print(f'Rango X: {df_ant_5km["x_m"].min():.0f} a {df_ant_5km["x_m"].max():.0f} m')
print(f'Rango Y: {df_ant_5km["y_m"].min():.0f} a {df_ant_5km["y_m"].max():.0f} m')

## Cell 6 · Array antennas (TX + RX)

In [ ]:
# TX array: isotrópico V-pol (simplificación, calibración two-slope corrige off-axis)
scene.tx_array = rt.PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='iso', polarization='V'
)

# RX array (cuando midamos cobertura con receivers individuales)
scene.rx_array = rt.PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='iso', polarization='V'
)
print('✅ TX/RX arrays configurados')

## Cell 7 · Función de coverage map por banda/altura

In [ ]:
def compute_coverage(band, z_m, df_antenas):
    """Corre Sionna RadioMapSolver para una banda y altura dadas.

    Usa co-ubicación UMTS/LTE: incluye TODAS las antenas del operador
    (no solo las LTE reportadas en OpenCellID).
    """
    freq_mhz = cfg.SYSTEMIC_BAND_FREQ_MHZ[band]
    scene.frequency = freq_mhz * 1e6
    print(f'\n▶ Banda {band} ({freq_mhz} MHz) · altura z={z_m} m')

    # Limpiar TX previos
    tx_names = [tx.name for tx in list(scene.transmitters.values())]
    for name in tx_names:
        scene.remove(name)

    # Agregar TX (co-ubicación: todas las antenas del operador contribuyen)
    n_tx = 0
    for idx, row in df_antenas.iterrows():
        operator = row['operator']
        # Cualquier operador con LTE en esta banda
        if not cfg.is_band_compatible_with_tech(band, 'LTE'):
            continue
        eirp = cfg.get_eirp_dbm(operator, 'LTE')
        try:
            tx = rt.Transmitter(
                name=f'tx_{int(row["id"])}',
                position=[float(row['x_m']), float(row['y_m']), float(row['z_m'])],
                power_dbm=float(eirp),
            )
            scene.add(tx)
            n_tx += 1
        except Exception as e:
            pass
    print(f'  TX agregados: {n_tx}')

    # Solver
    t0 = time.time()
    solver = rt.RadioMapSolver()
    rm = solver(
        scene=scene,
        max_depth=cfg.SIONNA_SOLVER_CONFIG['max_depth'],
        samples_per_tx=cfg.SIONNA_SOLVER_CONFIG['samples_per_tx'],
        cell_size=cfg.SIONNA_SOLVER_CONFIG['cell_size'],
        diffraction=cfg.SIONNA_SOLVER_CONFIG['diffraction'],
        specular_reflection=cfg.SIONNA_SOLVER_CONFIG['specular_reflection'],
        diffuse_reflection=cfg.SIONNA_SOLVER_CONFIG['diffuse_reflection'],
        refraction=cfg.SIONNA_SOLVER_CONFIG['refraction'],
        rx_height=z_m,
    )
    dt = time.time() - t0
    print(f'  Solver completado en {dt:.1f} s')
    return rm

## Cell 8 · Ejecutar las 6 corridas (3 bandas × 2 alturas)

In [ ]:
RESULTS = {}
for band in cfg.SYSTEMIC_PRIORITY_BANDS:           # B26, B28, B41
    for z in cfg.SIONNA_ANALYSIS_HEIGHTS_M:         # 1.5, 7.5
        key = f'{band}_z{z}'
        rm = compute_coverage(band, z, df_ant_5km)
        RESULTS[key] = rm
        # Guardar cada radio map intermedio (por si cae la sesión)
        np.save(f'{OUTPUTS}/rm_{key}.npy', rm.path_gain.numpy() if hasattr(rm, 'path_gain') else np.array([]))
print(f'\n✅ {len(RESULTS)} corridas completadas')

## Cell 9 · Aplicar calibración two-slope LOS/NLOS (ITU-R M.2412)

Offset heredado de la calibración Recreo: **+41.2 dB** aplicado a puntos en régimen NLOS (delta_raw ≥ 30 dB respecto a free-space de referencia). Ver F1/CALIBRACION_TRANSFERIDA_DE_RECREO.md sección 2.2.

In [ ]:
def calibrate_two_slope(raw_dbm, los_mask, nlos_offset=cfg.SYSTEMIC_NLOS_OFFSET_DB):
    """Aplica offset NLOS a puntos con raw signal < threshold. LOS sin corrección."""
    calibrated = np.where(los_mask, raw_dbm, raw_dbm + nlos_offset)
    return calibrated

# Clutter adicional según escenario central (peri-urbano Piñero)
CLUTTER_FLOOR = cfg.SITE_PINERO_CLUTTER_FLOOR_DB['central']   # 20 dB
CLUTTER_ALTURA = cfg.SITE_PINERO_CLUTTER_ALTURA_DB['central']  # 8.5 dB
print(f'Clutter aplicado: floor={CLUTTER_FLOOR} dB · altura={CLUTTER_ALTURA} dB')

# TODO: loop sobre RESULTS aplicando clutter según altura + two-slope
# (la forma exacta depende de la API Sionna 2.x: .path_gain o .rss)
# Placeholder calibrado para merge a CSV:
calibrated_maps = {}
for key, rm in RESULTS.items():
    band, z_str = key.rsplit('_z', 1)
    z = float(z_str)
    clutter = CLUTTER_FLOOR if z <= 2.0 else CLUTTER_ALTURA
    raw = rm.path_gain.numpy() if hasattr(rm, 'path_gain') else None
    if raw is None:
        continue
    raw_dbm = 10 * np.log10(np.maximum(raw.sum(axis=0), 1e-20)) + 45  # EIRP 45 dBm asumido
    los_mask = raw_dbm > -60  # heurística: puntos fuertes = LOS
    cal = calibrate_two_slope(raw_dbm, los_mask) - clutter - cfg.SYSTEMIC_ANTENNA_MISPOINTING_DB
    calibrated_maps[key] = cal
    print(f'  {key}: shape {cal.shape} · mediana {np.median(cal):.1f} dBm · p75 {np.percentile(cal, 75):.1f} dBm')

## Cell 10 · Exportar CSV + heatmaps PNG

In [ ]:
# CSV consolidado
rows = []
for key, cal in calibrated_maps.items():
    band, z_str = key.rsplit('_z', 1)
    ny, nx = cal.shape
    for i in range(ny):
        for j in range(nx):
            rows.append({
                'band': band,
                'z_m': float(z_str),
                'cell_x': j,
                'cell_y': i,
                'rx_dbm_calibrated': float(cal[i, j]),
            })
df_out = pd.DataFrame(rows)
csv_path = f'{OUTPUTS}/mediciones_sionna_coverage_pinero.csv'
df_out.to_csv(csv_path, index=False)
print(f'CSV exportado: {csv_path} ({len(df_out)} filas)')

# Heatmap compuesto 3x2
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for col, band in enumerate(cfg.SYSTEMIC_PRIORITY_BANDS):
    for row_idx, z in enumerate(cfg.SIONNA_ANALYSIS_HEIGHTS_M):
        key = f'{band}_z{z}'
        if key not in calibrated_maps:
            continue
        ax = axes[row_idx, col]
        cal = calibrated_maps[key]
        im = ax.imshow(cal, cmap='viridis', vmin=-120, vmax=-30, origin='lower')
        ax.set_title(f'{band} ({cfg.SYSTEMIC_BAND_FREQ_MHZ[band]} MHz) · z={z} m')
        plt.colorbar(im, ax=ax, label='Rx [dBm calibrado]')
plt.suptitle('Mini Penal A · UP N°8 Piñero · Cobertura celular (post-calibración two-slope)')
plt.tight_layout()
png_path = f'{OUTPUTS}/heatmaps_sionna_pinero.png'
plt.savefig(png_path, dpi=200, bbox_inches='tight')
print(f'Heatmap compuesto: {png_path}')
plt.show()

## Cell 11 · Stats resumen

In [ ]:
import json as _json
stats = {
    'scene': 'MiniPenalA_UP8_Pinero',
    'coordinate_ref_UP8': {'lat': UP8_LAT, 'lon': UP8_LON},
    'n_tx_used': int(df_ant_5km.shape[0]),
    'bands': cfg.SYSTEMIC_PRIORITY_BANDS,
    'heights_m': cfg.SIONNA_ANALYSIS_HEIGHTS_M,
    'clutter_scenario': 'central',
    'clutter_floor_db': CLUTTER_FLOOR,
    'clutter_altura_db': CLUTTER_ALTURA,
    'nlos_offset_db': cfg.SYSTEMIC_NLOS_OFFSET_DB,
    'solver_config': cfg.SIONNA_SOLVER_CONFIG,
    'per_band_medians_dbm': {},
    'calibration_source': 'transferred_from_recreo_IDR_april2026',
}
for key, cal in calibrated_maps.items():
    stats['per_band_medians_dbm'][key] = {
        'median': float(np.median(cal)),
        'p25': float(np.percentile(cal, 25)),
        'p75': float(np.percentile(cal, 75)),
        'max': float(np.max(cal)),
        'cells_above_m90_pct': float(100 * (cal > -90).sum() / cal.size),
    }
with open(f'{OUTPUTS}/sionna_stats_pinero.json', 'w') as f:
    _json.dump(stats, f, indent=2, ensure_ascii=False)
print('✅ Stats guardadas en sionna_stats_pinero.json')
print(_json.dumps(stats, indent=2))

## Cell 12 · Finalización + download outputs

Los outputs están en `/content/outputs/`. Al ser storage efímero de Colab, bajálos antes de cerrar la sesión.

```
/content/outputs/
├── heatmaps_sionna_pinero.png            (figura compuesta 3x2)
├── mediciones_sionna_coverage_pinero.csv (dataset espacial, ~30 MB)
├── sionna_stats_pinero.json              (métricas agregadas)
└── rm_{band}_z{h}.npy × 6                (radio maps raw)
```

La celda de abajo crea un `.zip` con todo y lo baja a tu computadora.

In [ ]:
import shutil
from google.colab import files

zip_path = '/content/PINERO_RF_outputs.zip'
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', OUTPUTS)
print(f'Zip creado: {zip_path}')
!ls -la {OUTPUTS}
print('\n⬇ Bajando zip...')
files.download(zip_path)